In [1]:
import duckdb
import pandas as pd
import plotly.express as px

DB_PATH = r"D:\lap_trinh\my_project\book-pipeline\data\warehouse\gold.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)

print("Kết nối DuckDB thành công")

Kết nối DuckDB thành công


In [2]:
queries = {
    # =========================================================
    # 01. DANH SÁCH BẢNG
    # =========================================================
    "01_tables": """
        select
            table_schema,
            table_name,
            table_type
        from information_schema.tables
        where table_schema = 'main'
        order by table_name
    """,

    # =========================================================
    # 02. SỐ DÒNG MỖI BẢNG
    # =========================================================
    "02_table_counts": """
        select 'dim_books' as table_name, count(*) as rows from dim_books
        union all
        select 'dim_authors', count(*) from dim_authors
        union all
        select 'dim_editions', count(*) from dim_editions
        union all
        select 'dim_tags', count(*) from dim_tags
        union all
        select 'bridge_book_tags', count(*) from bridge_book_tags
        union all
        select 'fact_book_metrics', count(*) from fact_book_metrics
        order by rows desc
    """,

    # =========================================================
    # 03. KIỂM TRA TRÙNG KHÓA
    # =========================================================
    "03_key_duplicates": """
        with bridge_keys as (
            select count(*) as unique_keys
            from (
                select distinct book_id, tag_id
                from bridge_book_tags
            )
        )
        select
            'dim_books.book_id' as key_name,
            count(*) as total_rows,
            count(distinct book_id) as unique_keys,
            count(*) - count(distinct book_id) as duplicate_rows
        from dim_books

        union all

        select
            'dim_authors.author_id',
            count(*),
            count(distinct author_id),
            count(*) - count(distinct author_id)
        from dim_authors

        union all

        select
            'dim_editions.edition_id',
            count(*),
            count(distinct edition_id),
            count(*) - count(distinct edition_id)
        from dim_editions

        union all

        select
            'dim_tags.tag_id',
            count(*),
            count(distinct tag_id),
            count(*) - count(distinct tag_id)
        from dim_tags

        union all

        select
            'fact_book_metrics.book_id',
            count(*),
            count(distinct book_id),
            count(*) - count(distinct book_id)
        from fact_book_metrics

        union all

        select
            'bridge_book_tags.(book_id,tag_id)',
            count(*),
            (select unique_keys from bridge_keys),
            count(*) - (select unique_keys from bridge_keys)
        from bridge_book_tags
    """,

    # =========================================================
    # 04. MISSING DATA TRONG DIM_BOOKS
    # =========================================================
    "04_books_missing": """
        select
            count(*) as total_books,

            count_if(title is null or trim(title) = '')
                as missing_title,

            count_if(author_names is null or trim(author_names) = '')
                as missing_author,

            count_if(description is null or trim(description) = '')
                as missing_description,

            count_if(genres is null or trim(genres) = '')
                as missing_genres,

            count_if(moods is null or trim(moods) = '')
                as missing_moods,

            count_if(rating is null)
                as missing_rating,

            count_if(pages is null)
                as missing_pages,

            count_if(release_year is null)
                as missing_release_year,

            count_if(chunk_text is null or trim(chunk_text) = '')
                as missing_chunk
        from dim_books
    """,

    # =========================================================
    # 05. TỶ LỆ MISSING
    # =========================================================
    "05_missing_percent": """
        select
            round(
                100.0 * count_if(
                    author_names is null or trim(author_names) = ''
                ) / count(*),
                2
            ) as missing_author_percent,

            round(
                100.0 * count_if(
                    description is null or trim(description) = ''
                ) / count(*),
                2
            ) as missing_description_percent,

            round(
                100.0 * count_if(
                    genres is null or trim(genres) = ''
                ) / count(*),
                2
            ) as missing_genres_percent,

            round(
                100.0 * count_if(
                    moods is null or trim(moods) = ''
                ) / count(*),
                2
            ) as missing_moods_percent,

            round(
                100.0 * count_if(rating is null) / count(*),
                2
            ) as missing_rating_percent
        from dim_books
    """,

    # =========================================================
    # 06. GIÁ TRỊ BẤT THƯỜNG
    # =========================================================
    "06_invalid_values": """
        select
            count(*) as total_books,

            count_if(rating < 0 or rating > 5)
                as invalid_rating,

            count_if(pages <= 0)
                as non_positive_pages,

            count_if(pages > 10000)
                as suspicious_pages,

            count_if(release_year < 1450)
                as suspicious_old_year,

            count_if(release_year > year(current_date) + 1)
                as suspicious_future_year,

            min(rating) as minimum_rating,
            max(rating) as maximum_rating,

            min(pages) as minimum_pages,
            max(pages) as maximum_pages,

            min(release_year) as minimum_year,
            max(release_year) as maximum_year
        from dim_books
    """,

    # =========================================================
    # 07. KIỂM TRA QUAN HỆ GIỮA CÁC BẢNG
    # =========================================================
    "07_relationships": """
        select
            'bridge_book_tags → dim_books' as relationship,
            count(*) as unmatched_rows
        from bridge_book_tags bt
        left join dim_books b
            on b.book_id = bt.book_id
        where b.book_id is null

        union all

        select
            'bridge_book_tags → dim_tags',
            count(*)
        from bridge_book_tags bt
        left join dim_tags t
            on t.tag_id = bt.tag_id
        where t.tag_id is null

        union all

        select
            'dim_editions → dim_books',
            count(*)
        from dim_editions e
        left join dim_books b
            on b.book_id = e.book_id
        where b.book_id is null

        union all

        select
            'fact_book_metrics → dim_books',
            count(*)
        from fact_book_metrics f
        left join dim_books b
            on b.book_id = f.book_id
        where b.book_id is null
    """,

    # =========================================================
    # 08. PHÂN BỐ LOẠI SÁCH
    # =========================================================
    "08_literary_types": """
        select
            coalesce(literary_type, 'Missing') as literary_type,
            count(*) as books,
            round(100.0 * count(*) / sum(count(*)) over (), 2)
                as percentage,
            round(avg(rating), 2) as average_rating,
            count_if(rating is null) as missing_rating
        from dim_books
        group by literary_type
        order by books desc
    """,

    # =========================================================
    # 09. PHÂN BỐ BOOK CATEGORY
    # =========================================================
    "09_book_categories": """
        select
            coalesce(book_category, 'Missing') as book_category,
            count(*) as books,
            round(100.0 * count(*) / sum(count(*)) over (), 2)
                as percentage,
            round(avg(rating), 2) as average_rating
        from dim_books
        group by book_category
        order by books desc
    """,

    # =========================================================
    # 10. PHÂN BỐ RATING
    # =========================================================
    "10_rating_distribution": """
        select
            case
                when rating is null then '00. Missing'
                when rating < 1 then '01. Under 1'
                when rating < 2 then '02. 1–1.99'
                when rating < 3 then '03. 2–2.99'
                when rating < 4 then '04. 3–3.99'
                when rating < 4.5 then '05. 4–4.49'
                else '06. 4.5–5'
            end as rating_group,

            count(*) as books,

            round(
                100.0 * count(*) / sum(count(*)) over (),
                2
            ) as percentage

        from dim_books
        group by 1
        order by 1
    """,

    # =========================================================
    # 11. THỐNG KÊ RATING
    # =========================================================
    "11_rating_summary": """
        select
            count(rating) as books_with_rating,
            round(avg(rating), 3) as average_rating,
            round(median(rating), 3) as median_rating,
            round(stddev(rating), 3) as rating_stddev,
            round(quantile_cont(rating, 0.25), 3) as rating_q1,
            round(quantile_cont(rating, 0.75), 3) as rating_q3,
            min(rating) as minimum_rating,
            max(rating) as maximum_rating
        from dim_books
    """,

    # =========================================================
    # 12. PHÂN BỐ SỐ TRANG
    # =========================================================
    "12_page_distribution": """
        select
            case
                when pages is null then '00. Missing'
                when pages <= 0 then '01. Invalid'
                when pages < 100 then '02. Under 100'
                when pages < 200 then '03. 100–199'
                when pages < 300 then '04. 200–299'
                when pages < 500 then '05. 300–499'
                when pages < 1000 then '06. 500–999'
                else '07. 1000+'
            end as page_group,

            count(*) as books,

            round(
                100.0 * count(*) / sum(count(*)) over (),
                2
            ) as percentage

        from dim_books
        group by 1
        order by 1
    """,

    # =========================================================
    # 13. THỐNG KÊ SỐ TRANG
    # =========================================================
    "13_page_summary": """
        select
            count(pages) as books_with_pages,
            round(avg(pages), 2) as average_pages,
            median(pages) as median_pages,
            quantile_cont(pages, 0.25) as pages_q1,
            quantile_cont(pages, 0.75) as pages_q3,
            min(pages) as minimum_pages,
            max(pages) as maximum_pages
        from dim_books
        where pages > 0
    """,

    # =========================================================
    # 14. SÁCH THEO THẬP KỶ
    # =========================================================
    "14_books_by_decade": """
        select
            cast(floor(release_year / 10) * 10 as integer)
                as decade,
            count(*) as books,
            round(avg(rating), 2) as average_rating
        from dim_books
        where release_year between 1450 and year(current_date) + 1
        group by 1
        order by decade
    """,

    # =========================================================
    # 15. COVERAGE TÁC GIẢ
    # =========================================================
    "15_author_coverage": """
        select
            count(*) as total_books,

            count_if(
                author_names is not null
                and trim(author_names) <> ''
            ) as books_with_authors,

            count_if(
                author_names is null
                or trim(author_names) = ''
            ) as books_without_authors,

            round(
                100.0 * count_if(
                    author_names is not null
                    and trim(author_names) <> ''
                ) / count(*),
                2
            ) as author_coverage_percent
        from dim_books
    """,

    # =========================================================
    # 16. TÁC GIẢ PHỔ BIẾN
    # Cột author_names đang là chuỗi tổng hợp nên đây chỉ là EDA sơ bộ
    # =========================================================
    "16_top_author_strings": """
        select
            author_names,
            count(*) as books,
            round(avg(rating), 2) as average_rating
        from dim_books
        where author_names is not null
          and trim(author_names) <> ''
        group by author_names
        order by books desc
        limit 30
    """,

    # =========================================================
    # 17. THỐNG KÊ EDITION
    # =========================================================
    "17_edition_summary": """
        with editions_per_book as (
            select
                book_id,
                count(*) as edition_count
            from dim_editions
            group by book_id
        )
        select
            (select count(*) from dim_editions)
                as total_editions,

            count(*) as books_with_editions,

            round(avg(edition_count), 2)
                as average_editions_per_book,

            median(edition_count)
                as median_editions_per_book,

            max(edition_count)
                as maximum_editions_per_book

        from editions_per_book
    """,

    # =========================================================
    # 18. SÁCH CÓ NHIỀU EDITION NHẤT
    # =========================================================
    "18_top_edition_books": """
        select
            b.book_id,
            b.title,
            count(*) as editions
        from dim_editions e
        join dim_books b
            on b.book_id = e.book_id
        group by b.book_id, b.title
        order by editions desc
        limit 30
    """,

    # =========================================================
    # 19. COVERAGE TAG THEO CATEGORY
    # Chỉ chạy được khi dim_tags đã chứa tag_category
    # =========================================================
    "19_tag_coverage": """
        select
            t.tag_category,

            count(*) as book_tag_pairs,

            count(distinct bt.book_id)
                as books,

            count(distinct bt.tag_id)
                as tags,

            round(
                100.0 * count(distinct bt.book_id)
                / (select count(*) from dim_books),
                2
            ) as book_coverage_percent

        from bridge_book_tags bt
        join dim_tags t
            on t.tag_id = bt.tag_id

        group by t.tag_category
        order by books desc
    """,

    # =========================================================
    # 20. TOP TAG
    # =========================================================
    "20_top_tags": """
        select
            t.tag_category,
            t.tag_name,
            count(distinct bt.book_id) as books
        from bridge_book_tags bt
        join dim_tags t
            on t.tag_id = bt.tag_id
        group by t.tag_category, t.tag_name
        order by books desc
        limit 50
    """,

    # =========================================================
    # 21. ĐỘ DÀI CHUNK
    # =========================================================
    "21_chunk_lengths": """
        select
            count(*) as total_books,

            count_if(length(chunk_text) < 50)
                as chunks_under_50,

            count_if(length(chunk_text) between 50 and 199)
                as chunks_50_199,

            count_if(length(chunk_text) between 200 and 499)
                as chunks_200_499,

            count_if(length(chunk_text) between 500 and 999)
                as chunks_500_999,

            count_if(length(chunk_text) between 1000 and 4999)
                as chunks_1000_4999,

            count_if(length(chunk_text) >= 5000)
                as chunks_5000_plus,

            round(avg(length(chunk_text)), 2)
                as average_length,

            median(length(chunk_text))
                as median_length,

            max(length(chunk_text))
                as maximum_length,

            count(distinct chunk_text)
                as unique_chunks
        from dim_books
    """,

    # =========================================================
    # 22. PHÂN LOẠI CHẤT LƯỢNG NỘI DUNG CHO RAG
    # =========================================================
    "22_rag_quality": """
        select
            case
                when (
                    lower(trim(title)) in (
                        '[unknown title]',
                        'unknown',
                        'untitled',
                        'no title'
                    )
                    or title is null
                )
                and (
                    description is null
                    or trim(description) = ''
                )
                then 'Poor'

                when description is not null
                 and length(trim(description)) >= 200
                then 'Rich'

                when (
                    author_names is not null
                    and trim(author_names) <> ''
                )
                or (
                    genres is not null
                    and trim(genres) <> ''
                )
                or (
                    moods is not null
                    and trim(moods) <> ''
                )
                then 'Metadata'

                else 'Poor'
            end as content_quality,

            count(*) as books,

            round(
                100.0 * count(*) / sum(count(*)) over (),
                2
            ) as percentage

        from dim_books
        group by 1
        order by books desc
    """,

    # =========================================================
    # 23. 30 CHUNK DÀI NHẤT
    # =========================================================
    "23_longest_chunks": """
        select
            book_id,
            title,
            length(chunk_text) as chunk_length,
            length(description) as description_length,
            left(description, 500) as description_preview
        from dim_books
        order by length(chunk_text) desc
        limit 30
    """,

    # =========================================================
    # 24. CHUNK TRÙNG NHIỀU NHẤT
    # =========================================================
    "24_duplicate_chunks": """
        select
            left(chunk_text, 300) as chunk_preview,
            count(*) as duplicated_books
        from dim_books
        where chunk_text is not null
          and trim(chunk_text) <> ''
        group by chunk_text
        having count(*) > 1
        order by duplicated_books desc
        limit 30
    """,

    # =========================================================
    # 25. TITLE TRÙNG NHIỀU NHẤT
    # Không đồng nghĩa là dữ liệu lỗi
    # =========================================================
    "25_duplicate_titles": """
        select
            lower(trim(title)) as normalized_title,
            count(*) as books,
            count(distinct author_names) as author_variants
        from dim_books
        where title is not null
          and trim(title) <> ''
        group by lower(trim(title))
        having count(*) > 1
        order by books desc
        limit 30
    """,

    # =========================================================
    # 26. DẤU HIỆU DESCRIPTION BẨN
    # =========================================================
    "26_description_quality": """
        select
            count(*) as total_books,

            count_if(
                description ilike '%http://%'
                or description ilike '%https://%'
            ) as descriptions_with_url,

            count_if(
                description like '%<div%'
                or description like '%<p>%'
                or description like '%<br%'
            ) as descriptions_with_html,

            count_if(length(description) >= 5000)
                as very_long_descriptions,

            count_if(
                description is not null
                and length(trim(description)) < 50
            ) as very_short_descriptions

        from dim_books
    """,

    # =========================================================
    # 27. ĐỘ MỚI DỮ LIỆU
    # =========================================================
    "27_freshness": """
        select
            min(updated_at) as oldest_updated_at,
            max(updated_at) as newest_updated_at,

            count_if(updated_at is null)
                as missing_updated_at,

            count_if(
                updated_at > current_timestamp + interval '1 day'
            ) as future_updated_at

        from dim_books
    """,

    # =========================================================
    # 28. TOP SÁCH RATING CAO CÓ ĐỦ LƯỢT ĐÁNH GIÁ
    # =========================================================
    "28_top_rated_books": """
        select
            book_id,
            title,
            author_names,
            rating,
            ratings_count,
            release_year
        from dim_books
        where rating is not null
          and ratings_count >= 100
        order by rating desc, ratings_count desc
        limit 30
    """,

    # =========================================================
    # 29. SÁCH ĐƯỢC ĐÁNH GIÁ NHIỀU NHẤT
    # =========================================================
    "29_most_rated_books": """
        select
            book_id,
            title,
            author_names,
            rating,
            ratings_count
        from dim_books
        where ratings_count is not null
        order by ratings_count desc
        limit 30
    """,

    # =========================================================
    # 30. MẪU SÁCH THIẾU TÁC GIẢ
    # =========================================================
    "30_missing_author_sample": """
        select
            book_id,
            title,
            description,
            genres,
            rating,
            ratings_count
        from dim_books
        where author_names is null
           or trim(author_names) = ''
        order by ratings_count desc nulls last
        limit 30
    """
}

In [3]:
results = {}
errors = {}

for name, sql in queries.items():
    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    try:
        df = con.sql(sql).df()
        results[name] = df
        display(df)

    except Exception as error:
        errors[name] = str(error)
        print("QUERY ERROR:")
        print(error)

print("\nEDA hoàn tất")
print("Thành công:", len(results))
print("Lỗi:", len(errors))


01_tables


,table_schema,table_name,table_type
0,main,bridge_book_authors,BASE TABLE
1,main,bridge_book_tags,BASE TABLE
2,main,dim_authors,BASE TABLE
3,main,dim_books,BASE TABLE
4,main,dim_editions,BASE TABLE
5,main,dim_tags,BASE TABLE
6,main,fact_book_metrics,BASE TABLE



02_table_counts


,table_name,rows
0,bridge_book_tags,2784267
1,dim_books,2536241
2,fact_book_metrics,2536241
3,dim_authors,978320
4,dim_editions,98122
5,dim_tags,41637



03_key_duplicates


...


29_most_rated_books


,book_id,title,author_names,rating,ratings_count
0,328491,Harry Potter and the Philosopher's Stone,J.K. Rowling,4.282109,10262
1,379760,1984,"George Orwell, Peter Hobley Davison, Shepard F...",4.237572,8811
2,88639,The Hunger Games,Suzanne Collins,4.136436,8165
3,427578,Project Hail Mary,Andy Weir,4.497482,7942
4,2459845,"The Hobbit, or There and Back Again",J.R.R. Tolkien,4.313468,7299
5,429306,Harry Potter and the Chamber of Secrets,J.K. Rowling,4.176859,7277
6,376219,Harry Potter and the Prisoner of Azkaban,J.K. Rowling,4.407561,7221
7,356467,Animal Farm,George Orwell,4.089372,6803
8,383236,Harry Potter and the Goblet of Fire,"J.K. Rowling, Mary GrandPré",4.365912,6712
9,428465,Harry Potter and the Order of the Phoenix,"J.K. Rowling, Mary GrandPré",4.291150,6294



30_missing_author_sample


,book_id,title,description,genres,rating,ratings_count
0,427524,The Invisible Life of Addie LaRue,"France, 1714: in a moment of desperation, a yo...","1738522998837, Fantasy, Fiction, Friendship, H...",4.126555,2331
1,374090,A Darker Shade of Magic,Kell is one of the last Atari - magicians with...,"Adventure, Dystopian, Fantasy, Fiction, Histor...",3.950569,1669
2,358274,A Gathering of Shadows,Four months have passed since the shadow stone...,"Adventure, Contests, Fantasy, Fiction, LGBTQ, ...",4.025219,912
3,148332,Consider Phlebas,The war raged across the galaxy. Billions had ...,"Adventure, Aliens, Audiobooks, Classics, Fanta...",3.655592,903
4,427451,The Girl with All the Gifts,Melanie is a very special girl. Dr Caldwell ca...,"Action & Adventure, Adventure, Apocalyptic fic...",4.015306,882
5,20256,The Player of Games,The Culture - a human/machine symbiotic societ...,"Adventure, Aliens, Audiobooks, Classics, Compu...",4.226790,754
6,10411,A Conjuring of Light,THE BALANCE OF POWER HAS FINALLY TIPPED....\nT...,"Adventure, Fantasy, Fantasy fiction, Fiction, ...",4.142573,754
7,427912,Vicious,"""In their senior year, a shared research inter...","Adventure, Fantasy, Fiction, Mystery, Occult, ...",4.058488,701
8,59640,Dead Until Dark,"For years, Charlaine Harris has delighted fans...","Cocktail servers, Fantasy, Fiction, Romance, S...",3.500000,653
9,427871,Vengeful,Magneto and Professor X. Superman and Lex Luth...,"Ability, Fantasy, Fiction, LGBTQ, Occult, Scie...",4.022822,482



EDA hoàn tất
Thành công: 30
Lỗi: 0


In [4]:
con.sql("""
    select
        count(*) as total_book_author_pairs,
        count(distinct book_id) as books_with_author,
        count(distinct author_id) as authors_used
    from bridge_book_authors
""").df()

,total_book_author_pairs,books_with_author,authors_used
0,2963234,2112813,822655


In [5]:
con.sql("""
    select
        'bridge_book_authors → dim_books' as relationship,
        count(*) as unmatched_rows
    from bridge_book_authors ba
    left join dim_books b
        on ba.book_id = b.book_id
    where b.book_id is null

    union all

    select
        'bridge_book_authors → dim_authors',
        count(*)
    from bridge_book_authors ba
    left join dim_authors a
        on ba.author_id = a.author_id
    where a.author_id is null
""").df()

,relationship,unmatched_rows
0,bridge_book_authors → dim_books,0
1,bridge_book_authors → dim_authors,0


In [6]:
con.sql("""
    select
        b.book_id,
        b.title,
        b.author_names as author_in_dim_books,
        string_agg(distinct a.name, ', ') as author_from_bridge,
        b.ratings_count
    from dim_books b
    join bridge_book_authors ba
        on ba.book_id = b.book_id
    join dim_authors a
        on a.author_id = ba.author_id
    where b.author_names is null
       or trim(b.author_names) = ''
    group by
        b.book_id,
        b.title,
        b.author_names,
        b.ratings_count
    order by b.ratings_count desc nulls last
    limit 30
""").df()

,book_id,title,author_in_dim_books,author_from_bridge,ratings_count
0,637078,The Journal of the Franklin Institute,,[Unknown author],0
1,2769090,Revolting Prostitutes,,[Unknown author],0


In [7]:
con.sql("""
    with bridge_coverage as (
        select distinct book_id
        from bridge_book_authors
    )

    select
        case
            when b.author_names is not null
             and trim(b.author_names) <> ''
             and bc.book_id is not null
                then 'Có ở cả dim_books và bridge'

            when (
                b.author_names is null
                or trim(b.author_names) = ''
            )
            and bc.book_id is not null
                then 'Thiếu ở dim_books nhưng có trong bridge'

            when b.author_names is not null
             and trim(b.author_names) <> ''
             and bc.book_id is null
                then 'Có ở dim_books nhưng thiếu trong bridge'

            else 'Thiếu ở cả hai'
        end as author_status,

        count(*) as books

    from dim_books b
    left join bridge_coverage bc
        on bc.book_id = b.book_id
    group by 1
    order by books desc
""").df()

,author_status,books
0,Có ở cả dim_books và bridge,2112811
1,Thiếu ở cả hai,423428
2,Thiếu ở dim_books nhưng có trong bridge,2


In [8]:
con.sql("""
        SELECT
    COUNT(*) FILTER (WHERE is_title_missing) AS missing_title,
    COUNT(*) FILTER (WHERE is_pages_suspicious) AS suspicious_pages,
    COUNT(*) FILTER (WHERE is_release_year_suspicious) AS suspicious_years
FROM main.dim_books;
        """).df()

,missing_title,suspicious_pages,suspicious_years
0,427,11908,67


In [9]:
con.sql("""
        SELECT
    COUNT(*) FILTER (WHERE is_title_missing) AS missing_title,
    COUNT(*) FILTER (WHERE is_pages_suspicious) AS suspicious_pages,
    COUNT(*) FILTER (WHERE is_release_year_suspicious) AS suspicious_years
FROM main.dim_books;""").df()

,missing_title,suspicious_pages,suspicious_years
0,427,11908,67


In [10]:
con.sql("""
       SELECT
    content_quality,
    COUNT(*) AS number_of_books,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM main.dim_books
GROUP BY content_quality
ORDER BY number_of_books DESC;""").df()

,content_quality,number_of_books,percentage
0,Metadata,1191505,46.98
1,Rich,1022502,40.32
2,Poor,322234,12.71


In [11]:
con.sql("""
       SELECT
    book_id,
    title,
    author_names,
    genres,
    content_quality,
    LEFT(chunk_text, 500) AS text_preview
FROM main.dim_books
WHERE content_quality = 'Rich'
LIMIT 10;""").df()

,book_id,title,author_names,genres,content_quality,text_preview
0,1826515,"Conan Omnibus, Vol. 3: Ancient Gods and Sorcerers","Richard Corben, Timothy Truman, Tomás Giorello","Adventure, Comics, Comics & Graphic Novels",Rich,"Title: Conan Omnibus, Vol. 3: Ancient Gods and..."
1,1825498,My Alien Beast,Stasia Black,"Aliens, Fantasy, Fiction, Science Fiction",Rich,Title: My Alien Beast\nAuthors: Stasia Black\n...
2,1825595,Room Fifteen: Making Her Obey,Brynn Paulin,Fiction,Rich,Title: Room Fifteen: Making Her Obey\nSubtitle...
3,1825598,Room Two: Love is Blind,Barbra Campbell,Fiction,Rich,Title: Room Two: Love is Blind\nSubtitle: Love...
4,1825653,I'll Never Call Him Dad Again,Caroline Darian,"Biography & Autobiography, Date rape drugs, No...",Rich,Title: I'll Never Call Him Dad Again\nSubtitle...
5,1825656,10 Leadership Virtues for Disruptive Times: Co...,"Ken Blanchard, Tom Ziglar",Business & Economics,Rich,Title: 10 Leadership Virtues for Disruptive Ti...
6,1825668,Blacker the Berry...,Wallace Thurman,Fiction,Rich,Title: Blacker the Berry...\nAuthors: Wallace ...
7,1825675,Marta Oulie,"Sigrid Undset, Tiina Nunnally","Classics, Fiction, History",Rich,"Title: Marta Oulie\nAuthors: Sigrid Undset, Ti..."
8,1825679,'The Holy Fox': The Life of Lord Halifax,Andrew Roberts,"Biography, Great Britain, Politics",Rich,Title: 'The Holy Fox': The Life of Lord Halifa...
9,1825681,Happiness Is . . .: Simple Steps to a Life of Joy,A.R. Bernard,"Christian life, Religion, Science Fiction",Rich,Title: Happiness Is . . .: Simple Steps to a L...


In [12]:
import os
import json
import boto3
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
    aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minioadmin"),
    region_name="us-east-1",
)

BUCKET = os.getenv("MINIO_BUCKET", "hardcover-lake")

print("Đã kết nối MinIO")

Đã kết nối MinIO


In [13]:
response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix="bronze/",
    Delimiter="/",
)

for item in response.get("CommonPrefixes", []):
    print(item["Prefix"])

bronze/authors/
bronze/books/
bronze/countries/
bronze/editions/
bronze/languages/
bronze/platforms/
bronze/publishers/
bronze/reading_formats/
bronze/series/
bronze/tags/


In [14]:
response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix="bronze/authors/",
    MaxKeys=10,
)

for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"])

bronze/authors/authors_full_000000000.jsonl 55620
bronze/authors/authors_full_offset0000000_batch001.jsonl 55642
bronze/authors/authors_full_offset0000100_batch002.jsonl 48216
bronze/authors/authors_full_offset0000200_batch003.jsonl 53716
bronze/authors/authors_full_offset0000300_batch004.jsonl 56233
bronze/authors/authors_full_offset0000400_batch005.jsonl 52012
bronze/authors/authors_full_offset0000500_batch006.jsonl 51283
bronze/authors/authors_full_offset0000600_batch007.jsonl 54059
bronze/authors/authors_full_offset0000700_batch008.jsonl 54980
bronze/authors/authors_full_offset0000800_batch009.jsonl 52182


In [15]:
TARGET_AUTHOR_ID = 1101812


def find_object_with_id(data, target_id):
    """Tìm object có trường id khớp target_id, kể cả JSON bị lồng."""

    if isinstance(data, dict):
        if str(data.get("id")) == str(target_id):
            return data

        for value in data.values():
            result = find_object_with_id(value, target_id)

            if result is not None:
                return result

    elif isinstance(data, list):
        for value in data:
            result = find_object_with_id(value, target_id)

            if result is not None:
                return result

    return None


paginator = s3.get_paginator("list_objects_v2")

found_author = None
found_file = None
files_checked = 0

for page in paginator.paginate(
    Bucket=BUCKET,
    Prefix="bronze/authors/",
):
    for obj in page.get("Contents", []):
        key = obj["Key"]

        if not key.endswith(".jsonl"):
            continue

        files_checked += 1

        response = s3.get_object(
            Bucket=BUCKET,
            Key=key,
        )

        for raw_line in response["Body"].iter_lines():
            if not raw_line:
                continue

            try:
                data = json.loads(raw_line.decode("utf-8"))
            except json.JSONDecodeError:
                continue

            result = find_object_with_id(
                data,
                TARGET_AUTHOR_ID,
            )

            if result is not None:
                found_author = result
                found_file = key
                break

        if found_author is not None:
            break

        if files_checked % 100 == 0:
            print("Đã kiểm tra:", files_checked, "files")

    if found_author is not None:
        break

Đã kiểm tra: 100 files
Đã kiểm tra: 200 files
Đã kiểm tra: 300 files
Đã kiểm tra: 400 files
Đã kiểm tra: 500 files
Đã kiểm tra: 600 files
Đã kiểm tra: 700 files
Đã kiểm tra: 800 files
Đã kiểm tra: 900 files
Đã kiểm tra: 1000 files
Đã kiểm tra: 1100 files
Đã kiểm tra: 1200 files
Đã kiểm tra: 1300 files
Đã kiểm tra: 1400 files
Đã kiểm tra: 1500 files
Đã kiểm tra: 1600 files
Đã kiểm tra: 1700 files
Đã kiểm tra: 1800 files
Đã kiểm tra: 1900 files
Đã kiểm tra: 2000 files
Đã kiểm tra: 2100 files
Đã kiểm tra: 2200 files
Đã kiểm tra: 2300 files
Đã kiểm tra: 2400 files
Đã kiểm tra: 2500 files
Đã kiểm tra: 2600 files
Đã kiểm tra: 2700 files
Đã kiểm tra: 2800 files
Đã kiểm tra: 2900 files
Đã kiểm tra: 3000 files
Đã kiểm tra: 3100 files
Đã kiểm tra: 3200 files
Đã kiểm tra: 3300 files
Đã kiểm tra: 3400 files
Đã kiểm tra: 3500 files
Đã kiểm tra: 3600 files
Đã kiểm tra: 3700 files
Đã kiểm tra: 3800 files
Đã kiểm tra: 3900 files
Đã kiểm tra: 4000 files
Đã kiểm tra: 4100 files
Đã kiểm tra: 4200 files
Đ

In [16]:
if found_author is None:
    print(
        f"Không tìm thấy author_id={TARGET_AUTHOR_ID} "
        f"trong {files_checked} file Bronze"
    )
else:
    print("Tìm thấy trong file:", found_file)
    print(
        json.dumps(
            found_author,
            ensure_ascii=False,
            indent=2,
        )
    )

Không tìm thấy author_id=1101812 trong 9990 file Bronze


In [17]:
con.sql("""
    select
        book_id, title, chunk_text,
        split_part(genres, ', ', 1) as primary_genre,
        rating, ratings_count, release_year
    from dim_books
    where content_quality = 'Rich'
      and ratings_count >= 50
    qualify row_number() over (
        partition by split_part(genres, ', ', 1)
        order by ratings_count desc
    ) <= 200
""").df()

,book_id,title,chunk_text,primary_genre,rating,ratings_count,release_year
0,552021,"Henry IV, Part 1","Title: Henry IV, Part 1\nSubtitle: Pt. 1\nAuth...",1367-1413,3.900000,65,1592
1,255674,Hamlet,Title: Hamlet\nSubtitle: Prince of Denmark\nAu...,1564-1616 - Texts,4.012812,1561,1602
2,382874,The God Delusion,Title: The God Delusion\nAuthors: Richard Dawk...,1735854268453,3.946402,681,2001
3,202496,Seveneves,Title: Seveneves\nAuthors: Neal Stephenson\nGe...,1735854311414,3.982823,1077,2015
4,258729,Anathem,Title: Anathem\nAuthors: Neal Stephenson\nGenr...,1735854313295,4.225649,616,2008
...,...,...,...,...,...,...,...
5920,817419,Swift and Saddled,Title: Swift and Saddled\nAuthors: Lyla Sage\n...,Fiction,3.744755,143,2024
5921,440284,Heat Wave,Title: Heat Wave\nAuthors: Richard Castle\nGen...,Fiction,3.380282,142,2008
5922,1084896,Forget Me Not,Title: Forget Me Not\nAuthors: Julie Soto\nGen...,Fiction,3.964789,142,2023
5923,817549,Behind the Net,Title: Behind the Net\nAuthors: Stephanie Arc...,Fiction,3.524648,142,2023
